<div dir="rtl">

# جلسه ۲۲: NumPy ۲
## NumPy II

**بخش ۳: محاسبات علمی و کار با داده** — دوره‌ی پایتون برای علوم داده

مدرس: دکتر حسین حق‌بین

</div>

<div dir="rtl">

## 🎯 اهداف جلسه

در پایان این جلسه باید بتوانید:

- به جای حلقه، محاسبه را روی **کل آرایه** بنویسید (برداری‌سازی) و توابع جهانی (<code dir="ltr">np.exp</code>، <code dir="ltr">np.where</code>، <code dir="ltr">np.clip</code>، ...) را به کار ببرید،
- قاعده‌های **broadcasting** را بدانید و شکل حاصل هر عملیات را (یا خطای آن را) پیش‌بینی کنید،
- با پارامتر <code dir="ltr">axis</code> و <code dir="ltr">keepdims</code> آمار هر سطر یا هر ستون را حساب کنید،
- با <code dir="ltr">np.random.default&#95;rng</code> اعداد تصادفی **تکرارپذیر** بسازید و داده را درست به هم بریزید،
- ضرب ماتریسی، حل دستگاه معادلات و کمترین مربعات را با <code dir="ltr">np.linalg</code> انجام دهید؛ همان عملیاتی که درون هر مدل یادگیری ماشین و هر لایه‌ی شبکه‌ی عصبی اجرا می‌شود.

> 📖 روش خواندن این جزوه و معنی علامت‌ها (💡 🔍 ❓ ⚠️ 📌 📝) در ابتدای جلسه‌ی ۱ توضیح داده شده است.

</div>

<div dir="rtl">

## فهرست مطالب

1. برداری‌سازی
2. broadcasting
3. توابع آماری
4. اعداد تصادفی
5. جبر خطی

</div>

<div dir="rtl">

## ۱. برداری‌سازی

### 💡 مفهوم

در جلسه‌ی ۲۱ دیدید که عملگرهای حسابی روی آرایه‌ها **عضو به عضو** اجرا می‌شوند. به نوشتن محاسبه روی کل آرایه، به جای حلقه روی تک‌تک عضوها، **برداری‌سازی (vectorization)** می‌گویند. دو مزیت دارد:

- **سرعت:** حلقه در زبان C و روی حافظه‌ی پیوسته اجرا می‌شود، نه در مفسر پایتون؛
- **خوانایی:** کد شبیه فرمول ریاضی می‌شود.

NumPy برای تقریباً هر تابع ریاضی یک **تابع جهانی (ufunc)** دارد که عضو به عضو روی آرایه کار می‌کند: <code dir="ltr">np.sqrt</code>، <code dir="ltr">np.exp</code>، <code dir="ltr">np.log</code>، <code dir="ltr">np.abs</code>، <code dir="ltr">np.sin</code>، ...

</div>

In [ ]:
import numpy as np

x = np.array([1, 4, 9, 16])
print(np.sqrt(x))
print(np.exp(np.array([0, 1, 2])).round(3))

prices = np.array([120.0, 80.0, 45.5])
qty = np.array([2, 5, 10])
print(prices * qty, (prices * qty).sum())

<div dir="rtl">

### 🔍 کد را بخوان

- <code dir="ltr">np.sqrt(x)</code> بدون هیچ حلقه‌ای از هر عضو جذر می‌گیرد: <code dir="ltr">[1. 2. 3. 4.]</code>. حاصل <code dir="ltr">float</code> است.
- <code dir="ltr">prices &#42; qty</code> همان «ضرب عضو به عضو» جلسه‌ی ۸ است که آنجا با <code dir="ltr">zip</code> و حلقه می‌نوشتیم؛ <code dir="ltr">.sum()</code> هم جمع کل را می‌دهد (<code dir="ltr">1095.0</code>).

### 💡 مقایسه‌ی سرعت

جمع مربعات یک میلیون عدد را یک بار با حلقه‌ی پایتون (generator expression، جلسه‌ی ۱۴) و یک بار برداری حساب می‌کنیم. زمان دقیق به کامپیوتر شما بستگی دارد؛ عدد مهم **نسبت** دو زمان است:

</div>

In [ ]:
import time

n = 1_000_000
values = list(range(n))
arr = np.arange(n)

t0 = time.perf_counter()
s1 = sum(v * v for v in values)
t_loop = time.perf_counter() - t0

t0 = time.perf_counter()
s2 = (arr * arr).sum()
t_vec = time.perf_counter() - t0

print(s1 == s2)
print(f"loop: {t_loop:.4f} s   vectorized: {t_vec:.4f} s   ratio: {t_loop / t_vec:.0f}x")

<div dir="rtl">

🔍 هر دو روش جواب یکسان می‌دهند (<code dir="ltr">True</code>)، اما نسخه‌ی برداری معمولاً **چندین تا ده‌ها برابر** سریع‌تر است. در یادگیری ماشین که داده میلیون‌ها عدد دارد و محاسبه هزاران بار تکرار می‌شود، همین تفاوت یعنی چند دقیقه در برابر چند ساعت.

### 💡 مقایسه، <code dir="ltr">np.where</code> و <code dir="ltr">np.clip</code>

عملگرهای مقایسه هم عضو به عضو کار می‌کنند و یک **آرایه‌ی بولی** می‌دهند (همان ماسک جلسه‌ی ۲۱). دو ابزار پرکاربرد:

- <code dir="ltr">np.where(cond, a, b)</code>: هر جا شرط درست است از <code dir="ltr">a</code> و هر جا نادرست است از <code dir="ltr">b</code> برمی‌دارد؛ نسخه‌ی برداری <code dir="ltr">a if cond else b</code>.
- <code dir="ltr">np.clip(x, lo, hi)</code>: هر عدد را به بازه‌ی <code dir="ltr">[lo, hi]</code> محدود می‌کند.

</div>

In [ ]:
a = np.array([-2, -1, 0, 3, 5])
print(a > 0)
print(np.where(a > 0, a, 0))      # ReLU
print(np.clip(a, -1, 3))
print((a > 0).sum(), (a > 0).mean())

<div dir="rtl">

### 🔍 کد را بخوان

- <code dir="ltr">np.where(a &gt; 0, a, 0)</code> عددهای منفی را صفر می‌کند: این دقیقاً تابع فعال‌سازی **ReLU** است که در جلسه‌ی ۱۹ با کلاس و حلقه نوشتیم. (<code dir="ltr">np.maximum(a, 0)</code> هم همین کار را می‌کند.)
- <code dir="ltr">np.clip</code> در یادگیری عمیق برای **بریدن گرادیان (gradient clipping)** و جلوگیری از <code dir="ltr">log(0)</code> استفاده می‌شود.
- در جمع، <code dir="ltr">True</code> مثل <code dir="ltr">1</code> حساب می‌شود؛ پس <code dir="ltr">(a &gt; 0).sum()</code> **تعداد** و <code dir="ltr">(a &gt; 0).mean()</code> **نسبت** عضوهای مثبت است (<code dir="ltr">2</code> و <code dir="ltr">0.4</code>). این الگو برای حساب کردن دقت (accuracy) بسیار رایج است: <code dir="ltr">(y&#95;pred == y&#95;true).mean()</code>.

### ⚠️ اشتباه رایج: <code dir="ltr">and</code> و <code dir="ltr">or</code> روی آرایه

</div>

In [ ]:
a = np.array([-2, -1, 0, 3, 5])
print(a[(a > -2) & (a < 4)])
print(a[(a > -2) and (a < 4)])

<div dir="rtl">

🔍 خط اول درست است: <code dir="ltr">&amp;</code> (و)، <code dir="ltr">&#124;</code> (یا) و <code dir="ltr">~</code> (نقیض) عضو به عضو کار می‌کنند و حاصل <code dir="ltr">[-1  0  3]</code> است. خط دوم خطای <code dir="ltr">ValueError: The truth value of an array ... is ambiguous</code> می‌دهد: <code dir="ltr">and</code> می‌خواهد کل آرایه را **یک** مقدار درست یا نادرست ببیند که معنی ندارد. پرانتزها هم لازم‌اند، چون اولویت <code dir="ltr">&amp;</code> از مقایسه بالاتر است (جلسه‌ی ۳).

### ❓ پیش‌بینی کن

</div>

In [ ]:
x = np.array([0.5, -1.5, 2.0, -0.2])
print(np.abs(x).max())
print(np.where(x < 0, -x, x))
print(np.round(1 / (1 + np.exp(-np.array([0.0, 100.0, -100.0]))), 3))

<div dir="rtl">

🔍 خط اول بزرگ‌ترین قدرمطلق است: <code dir="ltr">2.0</code>. خط دوم همان قدرمطلق را با <code dir="ltr">np.where</code> می‌سازد: <code dir="ltr">[0.5 1.5 2.  0.2]</code>. خط سوم تابع **سیگموید** را برداری حساب می‌کند: <code dir="ltr">[0.5 1.  0. ]</code>؛ برای <code dir="ltr">100</code> و <code dir="ltr">-100</code> حاصل به ۱ و ۰ بسیار نزدیک است. (برای <code dir="ltr">-1000</code> در محاسبه‌ی <code dir="ltr">np.exp(1000)</code> سرریز رخ می‌دهد و NumPy هشدار می‌دهد؛ مسئله‌ای که در سؤال ۳ برای softmax حل می‌کنیم.)

</div>

<div dir="rtl">

## ۲. broadcasting

### 💡 مفهوم

وقتی دو آرایه با **شکل‌های متفاوت** با هم جمع (یا ضرب و ...) می‌شوند، NumPy سعی می‌کند آرایه‌ی کوچک‌تر را بدون کپی واقعی «کش بدهد» تا هم‌شکل شوند. به این کار **broadcasting** می‌گویند. قاعده:

1. شکل‌ها را از **سمت راست** زیر هم بنویسید (اگر یکی بُعد کمتری دارد، از چپ با <code dir="ltr">1</code> پر می‌شود).
2. در هر بُعد، دو اندازه باید **برابر** باشند یا یکی از آن‌ها **<code dir="ltr">1</code>** باشد.
3. بُعدی که <code dir="ltr">1</code> است، به اندازه‌ی دیگری کش داده می‌شود. اگر در یک بُعد هیچ‌کدام برقرار نباشد، خطای <code dir="ltr">ValueError</code> رخ می‌دهد.

</div>

<p align="center"><img src="images/s22_broadcasting.png" width="760" alt="s22_broadcasting.png"></p>

<p align="center" dir="rtl"><i>جمع یک ستون با شکل <code dir="ltr">(3, 1)</code> و یک سطر با شکل <code dir="ltr">(1, 4)</code>: هر دو کش داده می‌شوند و حاصل شکل <code dir="ltr">(3, 4)</code> دارد.</i></p>

<div dir="rtl">

| شکل <code dir="ltr">A</code> | شکل <code dir="ltr">B</code> | شکل حاصل |
|---|---|---|
| <code dir="ltr">(3,)</code> | عدد | <code dir="ltr">(3,)</code> |
| <code dir="ltr">(2, 3)</code> | <code dir="ltr">(3,)</code> | <code dir="ltr">(2, 3)</code> |
| <code dir="ltr">(3, 1)</code> | <code dir="ltr">(1, 4)</code> | <code dir="ltr">(3, 4)</code> |
| <code dir="ltr">(5, 1)</code> | <code dir="ltr">(5,)</code> | <code dir="ltr">(5, 5)</code> ⚠️ |
| <code dir="ltr">(2, 3)</code> | <code dir="ltr">(2,)</code> | خطا |
| <code dir="ltr">(4, 3)</code> | <code dir="ltr">(4, 1)</code> | <code dir="ltr">(4, 3)</code> |

</div>

In [ ]:
M = np.array([[1, 2, 3],
              [4, 5, 6]])
print(M + 10)
print(M + np.array([100, 200, 300]))

col = np.array([[1], [2], [3]])        # shape (3, 1)
row = np.array([[10, 20, 30, 40]])     # shape (1, 4)
print((col + row).shape)
print(col * row)

<div dir="rtl">

### 🔍 کد را بخوان

- <code dir="ltr">M + 10</code>: عدد به همه‌ی عضوها اضافه می‌شود؛ ساده‌ترین حالت broadcasting.
- <code dir="ltr">M + np.array([100, 200, 300])</code>: شکل <code dir="ltr">(2, 3)</code> با <code dir="ltr">(3,)</code>؛ سطر به هر **سطر** <code dir="ltr">M</code> اضافه می‌شود (عدد اول به ستون اول و ...).
- <code dir="ltr">col &#42; row</code>: ستون <code dir="ltr">(3, 1)</code> در جهت افقی و سطر <code dir="ltr">(1, 4)</code> در جهت عمودی کش داده می‌شوند؛ حاصل یک **جدول ضرب** <code dir="ltr">3 × 4</code> است. به این حاصل **ضرب خارجی (outer product)** می‌گویند.

### 💡 کاربرد مهم: مرکز کردن ستون‌ها

ماتریس داده در یادگیری ماشین شکل <code dir="ltr">(n&#95;samples, n&#95;features)</code> دارد. میانگین هر ستون (ویژگی) یک بردار <code dir="ltr">(n&#95;features,)</code> است و با broadcasting از همه‌ی سطرها کم می‌شود:

</div>

In [ ]:
X = np.array([[1.0, 200.0],
              [2.0, 300.0],
              [3.0, 400.0]])
mu = X.mean(axis=0)
print(mu, mu.shape)
print(X - mu)

<div dir="rtl">

🔍 <code dir="ltr">mu</code> برابر <code dir="ltr">[  2. 300.]</code> با شکل <code dir="ltr">(2,)</code> است و <code dir="ltr">X - mu</code> از هر سطر کم می‌شود؛ حالا میانگین هر ستون صفر است. این همان کاری است که <code dir="ltr">StandardScaler</code> جلسه‌ی ۱۸ با حلقه انجام می‌داد؛ اینجا در یک خط.

### ⚠️ اشتباه رایج و خطرناک: <code dir="ltr">(n, 1)</code> در برابر <code dir="ltr">(n,)</code>

</div>

In [ ]:
y_true = np.array([1.0, 2.0, 3.0])            # shape (3,)
y_pred = np.array([[1.5], [2.0], [2.5]])      # shape (3, 1), e.g. output of X @ w with w of shape (d, 1)
err = y_pred - y_true
print(err.shape)
print((err ** 2).mean())
print(((y_pred.ravel() - y_true) ** 2).mean())

<div dir="rtl">

🔍 <code dir="ltr">y&#95;pred</code> ستونی است و <code dir="ltr">y&#95;true</code> یک‌بعدی؛ broadcasting بی‌صدا یک ماتریس <code dir="ltr">3 × 3</code> از **همه‌ی** تفاضل‌های دوبه‌دو می‌سازد و MSE غلط (<code dir="ltr">0.8333...</code>) به جای مقدار درست (<code dir="ltr">0.1666...</code>) حساب می‌شود. **هیچ خطایی** رخ نمی‌دهد و مدل فقط بد یاد می‌گیرد. این یکی از رایج‌ترین باگ‌های کدهای یادگیری ماشین است. عادت خوب: قبل از هر محاسبه‌ی خطا، <code dir="ltr">shape</code> را چاپ یا با یک شرط بررسی کنید (جلسه‌ی ۱۵).

### ❓ پیش‌بینی کن

شکل حاصل هر عبارت را بنویسید (یا بگویید خطا می‌دهد):

</div>

In [ ]:
A = np.ones((4, 3))
print((A + np.ones(3)).shape)
print((A * np.ones((4, 1))).shape)
print((np.ones((2, 1, 3)) + np.ones((4, 1))).shape)
print((A + np.ones(4)).shape)

<div dir="rtl">

🔍 به ترتیب <code dir="ltr">(4, 3)</code>، <code dir="ltr">(4, 3)</code> و <code dir="ltr">(2, 4, 3)</code>. در سومی شکل <code dir="ltr">(4, 1)</code> از چپ به <code dir="ltr">(1, 4, 1)</code> تبدیل می‌شود و هر بُعد با قاعده‌ی «برابر یا ۱» جور است. خط آخر خطا می‌دهد: آخرین بُعدها <code dir="ltr">3</code> و <code dir="ltr">4</code> هستند و هیچ‌کدام <code dir="ltr">1</code> نیست. (برای اضافه کردن یک عدد به هر **سطر**، باید <code dir="ltr">np.ones((4, 1))</code> بنویسید.)

</div>

<div dir="rtl">

## ۳. توابع آماری

### 💡 مفهوم

توابعی مثل <code dir="ltr">sum</code>، <code dir="ltr">mean</code>، <code dir="ltr">std</code>، <code dir="ltr">min</code>، <code dir="ltr">max</code>، <code dir="ltr">argmax</code> اگر بدون آرگومان صدا زده شوند، روی **کل** آرایه حساب می‌شوند. با <code dir="ltr">axis</code> مشخص می‌کنیم در کدام جهت «جمع شوند»:

- <code dir="ltr">axis=0</code>: در جهت **سطرها** جمع می‌شود (سطرها از بین می‌روند)؛ حاصل یک عدد برای هر **ستون**،
- <code dir="ltr">axis=1</code>: در جهت **ستون‌ها** جمع می‌شود؛ حاصل یک عدد برای هر **سطر**،
- <code dir="ltr">keepdims=True</code>: بُعد جمع‌شده با اندازه‌ی <code dir="ltr">1</code> حفظ می‌شود تا حاصل برای broadcasting آماده باشد.

</div>

In [ ]:
M = np.array([[1, 2, 3],
              [4, 5, 6]])
print(M.sum(), M.sum(axis=0), M.sum(axis=1))
print(M.max(axis=0), M.argmax(), M.argmax(axis=1))
print(M.mean(axis=1, keepdims=True))
print(M / M.sum(axis=1, keepdims=True))

<div dir="rtl">

### 🔍 کد را بخوان

- <code dir="ltr">M.sum(axis=0)</code> جمع هر ستون است: <code dir="ltr">[5 7 9]</code>؛ <code dir="ltr">M.sum(axis=1)</code> جمع هر سطر: <code dir="ltr">[ 6 15]</code>.
- <code dir="ltr">M.argmax()</code> بدون <code dir="ltr">axis</code> اندیس بزرگ‌ترین عضو را در آرایه‌ی **صاف‌شده** می‌دهد: <code dir="ltr">5</code>. <code dir="ltr">M.argmax(axis=1)</code> برای هر سطر اندیس ستونِ بزرگ‌ترین عضو را می‌دهد: <code dir="ltr">[2 2]</code>؛ این همان «کلاس پیش‌بینی‌شده» برای هر نمونه است.
- با <code dir="ltr">keepdims=True</code> جمع سطرها شکل <code dir="ltr">(2, 1)</code> دارد؛ پس <code dir="ltr">M / M.sum(axis=1, keepdims=True)</code> هر سطر را بر جمع **همان سطر** تقسیم می‌کند و جمع هر سطر حاصل <code dir="ltr">1</code> می‌شود. بدون <code dir="ltr">keepdims</code> شکل <code dir="ltr">(2,)</code> با <code dir="ltr">(2, 3)</code> جور نمی‌شد.

### 💡 مرتب‌سازی، جمع تجمعی، صدک و واریانس

</div>

In [ ]:
v = np.array([3, 1, 2, 5])
print(np.argsort(v))
print(v[np.argsort(v)[::-1]])
print(np.cumsum(v))
print(np.percentile(v, 50), np.quantile(v, [0.25, 0.75]))
print(np.var(v), np.var(v, ddof=1))

<div dir="rtl">

### 🔍 کد را بخوان

- <code dir="ltr">np.argsort(v)</code> **اندیس‌هایی** را می‌دهد که آرایه را مرتب می‌کنند: <code dir="ltr">[1 2 0 3]</code> (کوچک‌ترین عضو در اندیس ۱ است). با <code dir="ltr">[::-1]</code> ترتیب نزولی به دست می‌آید؛ برای پیدا کردن «k مورد برتر» (top-k) پرکاربرد است.
- <code dir="ltr">np.cumsum</code> جمع تجمعی است: <code dir="ltr">[ 3  4  6 11]</code>.
- <code dir="ltr">np.percentile(v, 50)</code> میانه است (<code dir="ltr">2.5</code>) و <code dir="ltr">np.quantile</code> همان را با کسر بین ۰ و ۱ می‌گیرد: چارک‌ها <code dir="ltr">[1.75 3.5 ]</code>.
- <code dir="ltr">np.var(v)</code> بر <code dir="ltr">n</code> تقسیم می‌کند (<code dir="ltr">2.1875</code>) و <code dir="ltr">np.var(v, ddof=1)</code> بر <code dir="ltr">n - 1</code> (<code dir="ltr">2.9166...</code>)؛ دومی برآورد **نااریب** واریانس از روی نمونه است. <code dir="ltr">std</code> هم همین پارامتر را دارد. (Pandas به‌طور پیش‌فرض <code dir="ltr">ddof=1</code> به کار می‌برد؛ در جلسه‌ی ۲۳ دوباره این را می‌بینید.)

### ⚠️ اشتباه رایج: جهت <code dir="ltr">axis</code>

بسیاری فکر می‌کنند <code dir="ltr">axis=0</code> یعنی «برای هر سطر». برعکس است: <code dir="ltr">axis=0</code> یعنی **محور سطرها از بین می‌رود** و برای هر ستون یک عدد باقی می‌ماند. راه ساده‌ی یادآوری: شکل حاصل را بنویسید؛ بُعدی که <code dir="ltr">axis</code> به آن اشاره می‌کند از شکل حذف می‌شود.

### ❓ پیش‌بینی کن

</div>

In [ ]:
scores = np.array([[0.1, 0.7, 0.2],
                   [0.5, 0.3, 0.2],
                   [0.2, 0.2, 0.6]])
labels = np.array([1, 0, 1])
pred = scores.argmax(axis=1)
print(pred, (pred == labels).mean())
print(scores.max(axis=1).round(2), scores.sum(axis=0).round(2))

<div dir="rtl">

🔍 پیش‌بینی هر سطر کلاس با بیشترین امتیاز است: <code dir="ltr">[1 0 2]</code>. دو پیش‌بینی از سه درست است؛ دقت <code dir="ltr">0.666...</code>. بیشینه‌ی هر سطر <code dir="ltr">[0.7 0.5 0.6]</code> و جمع هر ستون <code dir="ltr">[0.8 1.2 1. ]</code> است.

</div>

<div dir="rtl">

## ۴. اعداد تصادفی

### 💡 مفهوم

در جلسه‌ی ۱۷ با ماژول <code dir="ltr">random</code> و <code dir="ltr">random.seed</code> کار کردید. NumPy مولد خودش را دارد که آرایه‌ای از اعداد تصادفی را یک‌جا می‌سازد. روش توصیه‌شده‌ی امروزی ساختن یک **مولد (Generator)** با <code dir="ltr">np.random.default&#95;rng(seed)</code> است:

| متد | کار |
|---|---|
| <code dir="ltr">rng.random(size)</code> | عدد اعشاری یکنواخت در <code dir="ltr">[0, 1)</code> |
| <code dir="ltr">rng.integers(low, high, size)</code> | عدد صحیح در <code dir="ltr">[low, high)</code> |
| <code dir="ltr">rng.normal(mean, std, size)</code> | توزیع نرمال |
| <code dir="ltr">rng.choice(a, size, replace=True)</code> | انتخاب تصادفی از عضوهای <code dir="ltr">a</code> |
| <code dir="ltr">rng.permutation(n)</code> | یک جای‌گشت تصادفی از <code dir="ltr">0</code> تا <code dir="ltr">n - 1</code> (آرایه‌ی جدید) |
| <code dir="ltr">rng.shuffle(arr)</code> | به هم ریختن خود <code dir="ltr">arr</code> (درجا) |

</div>

In [ ]:
rng = np.random.default_rng(42)
print(rng.random(3).round(4))
print(rng.integers(1, 7, size=5))
print(rng.normal(0, 1, size=3).round(4))

rng = np.random.default_rng(42)
print(rng.choice(["a", "b", "c"], size=4))
print(rng.permutation(5))

<div dir="rtl">

### 🔍 کد را بخوان

- با <code dir="ltr">seed=42</code> خروجی‌ها روی هر کامپیوتری یکسان‌اند: سه عدد اول <code dir="ltr">[0.774  0.4389 0.8586]</code> و پنج تاس <code dir="ltr">[1 5 2 1 4]</code>.
- <code dir="ltr">rng.integers(1, 7)</code> مثل <code dir="ltr">range</code>، <code dir="ltr">7</code> را شامل نمی‌شود؛ پس اعداد ۱ تا ۶ (تاس) تولید می‌شوند. (برخلاف <code dir="ltr">random.randint(1, 6)</code> جلسه‌ی ۱۷ که هر دو سر را شامل است.)
- مولد را دوباره با همان <code dir="ltr">seed</code> ساختیم؛ پس دنباله از اول تکرار شد.

### 💡 تکرارپذیری

هر فراخوانی مولد را **جلو می‌برد**؛ دو مولد با یک <code dir="ltr">seed</code> دنباله‌ی یکسان تولید می‌کنند:

</div>

In [ ]:
r1 = np.random.default_rng(7)
r2 = np.random.default_rng(7)
print(r1.integers(0, 100, 3), r2.integers(0, 100, 3))
print(r1.integers(0, 100, 3))

<div dir="rtl">

🔍 دو خروجی اول یکسان‌اند (<code dir="ltr">[94 62 68]</code>)، اما فراخوانی دوم <code dir="ltr">r1</code> عددهای تازه می‌دهد (<code dir="ltr">[89 57 77]</code>). در کدهای قدیمی‌تر <code dir="ltr">np.random.seed(0)</code> و <code dir="ltr">np.random.rand(...)</code> را می‌بینید که یک مولد **سراسری** را به کار می‌برند؛ کار می‌کند، اما مولد جداگانه امن‌تر است چون هیچ کد دیگری حالت آن را عوض نمی‌کند.

### ⚠️ اشتباه رایج: به هم ریختن جداگانه‌ی <code dir="ltr">X</code> و <code dir="ltr">y</code>

اگر ویژگی‌ها و برچسب‌ها را جداگانه <code dir="ltr">shuffle</code> کنید، رابطه‌ی هر نمونه با برچسبش از بین می‌رود و مدل روی داده‌ی بی‌معنی آموزش می‌بیند. الگوی درست: **یک** جای‌گشت از اندیس‌ها بسازید و هر دو را با آن اندیس‌گذاری کنید (اندیس‌گذاری فانتزی، جلسه‌ی ۲۱):

</div>

In [ ]:
X = np.array([[1, 1], [2, 2], [3, 3], [4, 4]])
y = np.array([10, 20, 30, 40])
rng = np.random.default_rng(0)
idx = rng.permutation(len(y))
print(idx)
print(X[idx], y[idx])

<div dir="rtl">

🔍 سطر <code dir="ltr">i</code> از <code dir="ltr">X</code> همچنان با عضو <code dir="ltr">i</code> از <code dir="ltr">y</code> جفت است، چون هر دو با یک <code dir="ltr">idx</code> مرتب شده‌اند. تقسیم آموزش و آزمون جلسه‌ی ۸ هم با همین الگو برداری می‌شود: <code dir="ltr">train = idx[n&#95;test:]</code>، <code dir="ltr">test = idx[:n&#95;test]</code>.

### ❓ پیش‌بینی کن

بدون اجرا: آیا خروجی دو خط زیر یکسان است؟ چرا؟

</div>

In [ ]:
rng = np.random.default_rng(1)
a = rng.random(2)
b = rng.random(2)
print(np.array_equal(a, b))
print(np.array_equal(np.random.default_rng(1).random(2), a))

<div dir="rtl">

🔍 خط اول <code dir="ltr">False</code> است: <code dir="ltr">b</code> از ادامه‌ی همان دنباله آمده است. خط دوم <code dir="ltr">True</code> است: یک مولد تازه با همان <code dir="ltr">seed</code> دقیقاً همان دو عدد اول (<code dir="ltr">a</code>) را تولید می‌کند.

</div>

<div dir="rtl">

## ۵. جبر خطی

### 💡 مفهوم

تقریباً همه‌ی مدل‌های یادگیری ماشین در دل خود ضرب ماتریسی انجام می‌دهند. ابزارهای اصلی:

| عملیات | NumPy |
|---|---|
| ضرب ماتریسی | <code dir="ltr">A @ B</code> یا <code dir="ltr">np.dot(A, B)</code> |
| ترانهاده | <code dir="ltr">A.T</code> |
| وارون | <code dir="ltr">np.linalg.inv(A)</code> |
| حل <code dir="ltr">A x = b</code> | <code dir="ltr">np.linalg.solve(A, b)</code> |
| دترمینان | <code dir="ltr">np.linalg.det(A)</code> |
| طول (نرم) بردار | <code dir="ltr">np.linalg.norm(v)</code> |
| مقدار و بردار ویژه (ماتریس متقارن) | <code dir="ltr">np.linalg.eigh(A)</code> |
| کمترین مربعات | <code dir="ltr">np.linalg.lstsq(A, b, rcond=None)</code> |

قاعده‌ی شکل در ضرب ماتریسی: <code dir="ltr">(m, k) @ (k, n)</code> شکل <code dir="ltr">(m, n)</code> می‌دهد؛ بُعدهای **وسط** باید برابر باشند.

</div>

In [ ]:
X = np.array([[1.0, 2.0],
              [3.0, 4.0]])
print(X * X)
print(X @ X)
print(X.T)

u = np.array([1.0, 2.0, 3.0])
v = np.array([4.0, 5.0, 6.0])
print(u @ v, np.linalg.norm(np.array([3.0, 4.0])))

<div dir="rtl">

### 🔍 کد را بخوان

- <code dir="ltr">X &#42; X</code> ضرب **عضو به عضو** است (<code dir="ltr">[[ 1. 4.] [ 9. 16.]]</code>) و <code dir="ltr">X @ X</code> ضرب **ماتریسی** (<code dir="ltr">[[ 7. 10.] [15. 22.]]</code>).
- برای دو بردار یک‌بعدی، <code dir="ltr">u @ v</code> ضرب داخلی است: <code dir="ltr">32.0</code>؛ همان تابع <code dir="ltr">dot</code> که در جلسه‌ی ۸ با حلقه نوشتید.
- <code dir="ltr">norm</code> طول اقلیدسی است: طول <code dir="ltr">[3, 4]</code> برابر <code dir="ltr">5.0</code>.

### 💡 حل دستگاه معادلات

دستگاه <code dir="ltr">2x + y = 3</code> و <code dir="ltr">x + 3y = 5</code> را به شکل <code dir="ltr">A x = b</code> می‌نویسیم:

</div>

In [ ]:
A = np.array([[2.0, 1.0],
              [1.0, 3.0]])
b = np.array([3.0, 5.0])

x = np.linalg.solve(A, b)
print(x)
print(A @ x)
print(np.linalg.inv(A))
print(round(np.linalg.det(A), 6))

vals, vecs = np.linalg.eigh(A)
print(vals.round(4))

<div dir="rtl">

### 🔍 کد را بخوان

- جواب <code dir="ltr">x = 0.8</code> و <code dir="ltr">y = 1.4</code> است و <code dir="ltr">A @ x</code> دوباره <code dir="ltr">b</code> را می‌دهد (راه خوبی برای بررسی جواب).
- <code dir="ltr">det</code> برابر <code dir="ltr">5</code> است؛ اگر صفر بود، ماتریس وارون نداشت و دستگاه جواب یکتا نداشت. <code dir="ltr">round</code> لازم است چون محاسبه‌ی اعشاری عددی مثل <code dir="ltr">5.000000000000001</code> می‌دهد (جلسه‌ی ۲).
- <code dir="ltr">eigh</code> مقدارهای ویژه‌ی ماتریس متقارن را به ترتیب صعودی می‌دهد: <code dir="ltr">[1.382 3.618]</code>. مقدار و بردار ویژه پایه‌ی روش‌هایی مثل **PCA** هستند. (برای ماتریس غیرمتقارن <code dir="ltr">np.linalg.eig</code> به کار می‌رود که ممکن است عدد مختلط برگرداند.)

### ⚠️ اشتباه رایج: <code dir="ltr">inv(A) @ b</code> به جای <code dir="ltr">solve(A, b)</code>

از نظر ریاضی <code dir="ltr">x = A⁻¹ b</code> است، اما ساختن وارون هم **کندتر** است و هم خطای گرد کردن بیشتری دارد؛ به‌خصوص وقتی ماتریس به حالت تکین نزدیک است. همیشه برای حل دستگاه از <code dir="ltr">np.linalg.solve</code> (و برای دستگاه‌های بیش‌ازحد معادله از <code dir="ltr">lstsq</code>) استفاده کنید.

### ❓ پیش‌بینی کن

شکل حاصل یا خطا را بنویسید:

</div>

In [ ]:
W = np.ones((3, 2))
x = np.ones(3)
B = np.ones((4, 3))
print((x @ W).shape)
print((B @ W).shape)
print((W.T @ B.T).shape)
print((W @ B).shape)

<div dir="rtl">

🔍 <code dir="ltr">(2,)</code>، <code dir="ltr">(4, 2)</code> و <code dir="ltr">(2, 4)</code>. بردار یک‌بعدی <code dir="ltr">(3,)</code> در ضرب ماتریسی مثل یک سطر رفتار می‌کند. خط آخر خطا می‌دهد: <code dir="ltr">(3, 2) @ (4, 3)</code>؛ بُعدهای وسط (<code dir="ltr">2</code> و <code dir="ltr">4</code>) برابر نیستند. (<code dir="ltr">W.T @ B.T</code> ترانهاده‌ی <code dir="ltr">B @ W</code> است: <code dir="ltr">(A B)ᵀ = Bᵀ Aᵀ</code>.)

</div>

<div dir="rtl">

## 📌 آنچه باید بلد باشید

- برداری‌سازی و ufuncها (<code dir="ltr">np.sqrt</code>، <code dir="ltr">np.exp</code>، <code dir="ltr">np.log</code>، <code dir="ltr">np.abs</code>)؛ چرا از حلقه سریع‌تر است.
- مقایسه و ماسک بولی، <code dir="ltr">&amp;</code> / <code dir="ltr">&#124;</code> / <code dir="ltr">~</code> با پرانتز؛ چرا <code dir="ltr">and</code> روی آرایه خطا می‌دهد؛ <code dir="ltr">(mask).sum()</code> و <code dir="ltr">(mask).mean()</code>.
- <code dir="ltr">np.where</code> (ReLU)، <code dir="ltr">np.clip</code>، <code dir="ltr">np.maximum</code>.
- سه قاعده‌ی broadcasting؛ پیش‌بینی شکل حاصل یا خطا؛ خطر <code dir="ltr">(n, 1)</code> در برابر <code dir="ltr">(n,)</code>.
- <code dir="ltr">axis=0</code> و <code dir="ltr">axis=1</code> و <code dir="ltr">keepdims</code>؛ نرمال کردن هر سطر یا هر ستون.
- <code dir="ltr">argmax</code>، <code dir="ltr">argsort</code> (top-k)، <code dir="ltr">cumsum</code>، <code dir="ltr">percentile</code> / <code dir="ltr">quantile</code>، <code dir="ltr">var</code> و <code dir="ltr">std</code> با <code dir="ltr">ddof</code>.
- <code dir="ltr">np.random.default&#95;rng(seed)</code> و متدهای آن؛ تکرارپذیری؛ به هم ریختن <code dir="ltr">X</code> و <code dir="ltr">y</code> با یک جای‌گشت.
- <code dir="ltr">@</code> در برابر <code dir="ltr">&#42;</code>؛ قاعده‌ی شکل ضرب ماتریسی؛ <code dir="ltr">solve</code> به جای <code dir="ltr">inv</code>؛ <code dir="ltr">det</code>، <code dir="ltr">norm</code>، <code dir="ltr">eigh</code>، <code dir="ltr">lstsq</code>.

</div>

<div dir="rtl">

## 📝 سؤالات مرور / نمونه سؤال امتحان

> اول خودتان جواب دهید (روی کاغذ، بدون اجرای کد)، بعد پاسخ را باز کنید و در آخر کد را اجرا کنید.

### سؤال ۱ — شکل‌ها

فرض کنید <code dir="ltr">X = np.ones((5, 3))</code>، <code dir="ltr">w = np.ones(3)</code>، <code dir="ltr">c = np.ones((5, 1))</code> و <code dir="ltr">M = np.ones((3, 3))</code>. شکل حاصل هر عبارت را بنویسید یا بگویید خطا می‌دهد.

| عبارت | شکل؟ |
|---|---|
| <code dir="ltr">X @ w</code> | |
| <code dir="ltr">X + w</code> | |
| <code dir="ltr">X + c</code> | |
| <code dir="ltr">X.sum(axis=0)</code> | |
| <code dir="ltr">X.mean(axis=1, keepdims=True)</code> | |
| <code dir="ltr">X @ M @ w</code> | |
| <code dir="ltr">(X @ w) - c</code> | |
| <code dir="ltr">X.T @ X</code> | |
| <code dir="ltr">X @ c</code> | |

<details>
<summary><b>پاسخ</b></summary>

| عبارت | شکل |
|---|---|
| <code dir="ltr">X @ w</code> | <code dir="ltr">(5,)</code> |
| <code dir="ltr">X + w</code> | <code dir="ltr">(5, 3)</code> |
| <code dir="ltr">X + c</code> | <code dir="ltr">(5, 3)</code> |
| <code dir="ltr">X.sum(axis=0)</code> | <code dir="ltr">(3,)</code> |
| <code dir="ltr">X.mean(axis=1, keepdims=True)</code> | <code dir="ltr">(5, 1)</code> |
| <code dir="ltr">X @ M @ w</code> | <code dir="ltr">(5,)</code> |
| <code dir="ltr">(X @ w) - c</code> | <code dir="ltr">(5, 5)</code>: همان باگ بخش ۲؛ بردار <code dir="ltr">(5,)</code> با ستون <code dir="ltr">(5, 1)</code> یک ماتریس همه‌ی تفاضل‌ها می‌سازد |
| <code dir="ltr">X.T @ X</code> | <code dir="ltr">(3, 3)</code>: ماتریسی که در معادلات نرمال (سؤال ۵) به کار می‌رود |
| <code dir="ltr">X @ c</code> | خطا: <code dir="ltr">(5, 3) @ (5, 1)</code>؛ بُعدهای وسط <code dir="ltr">3</code> و <code dir="ltr">5</code> برابر نیستند |

</details>

### سؤال ۲ — استانداردسازی برداری

</div>

```python
X = np.array([[1.0, 10.0],
              [2.0, 20.0],
              [3.0, 30.0],
              [6.0, 40.0]])
mu = X.mean(axis=0)
sd = X.std(axis=0)
Z = (X - mu) / sd
```

<div dir="rtl">

الف) <code dir="ltr">mu</code>، <code dir="ltr">sd</code> و ستون اول <code dir="ltr">Z</code> را حساب کنید (<code dir="ltr">sd</code> را تا دو رقم اعشار). میانگین و انحراف معیار هر ستون <code dir="ltr">Z</code> چقدر است؟

ب) این سه خط را با کلاس <code dir="ltr">StandardScaler</code> جلسه‌ی ۱۸ مقایسه کنید: کدام خط معادل <code dir="ltr">fit</code> و کدام معادل <code dir="ltr">transform</code> است؟ چرا برای داده‌ی آزمون باید از همان <code dir="ltr">mu</code> و <code dir="ltr">sd</code> داده‌ی آموزش استفاده کرد؟

ج) اگر بخواهیم هر **سطر** را (به جای هر ستون) طوری مقیاس کنیم که طولش <code dir="ltr">1</code> شود، کد را بنویسید. چرا بدون <code dir="ltr">keepdims=True</code> خطا می‌گیرید؟

د) اگر یک ستون مقدار ثابت داشته باشد، چه اتفاقی می‌افتد و چطور از آن جلوگیری می‌کنید؟

<details>
<summary><b>پاسخ</b></summary>

الف) <code dir="ltr">mu = [3., 25.]</code>، <code dir="ltr">sd ≈ [1.87, 11.18]</code> (<code dir="ltr">np.std</code> به‌طور پیش‌فرض <code dir="ltr">ddof=0</code>). ستون اول <code dir="ltr">Z</code> برابر <code dir="ltr">[-1.069, -0.535, 0., 1.604]</code> است. میانگین هر ستون <code dir="ltr">Z</code> صفر و انحراف معیار آن <code dir="ltr">1</code> است.

ب) دو خط اول (<code dir="ltr">mu</code> و <code dir="ltr">sd</code>) معادل <code dir="ltr">fit</code> هستند و خط سوم معادل <code dir="ltr">transform</code>. اگر <code dir="ltr">mu</code> و <code dir="ltr">sd</code> را از داده‌ی آزمون حساب کنیم، اطلاعات داده‌ی آزمون وارد آماده‌سازی شده است (**نشت داده**) و نمونه‌ی آزمون جدید به تنهایی هیچ آماری برای مقیاس کردن ندارد. مدل با مقیاس داده‌ی آموزش یاد گرفته و همان مقیاس باید روی هر داده‌ی تازه اعمال شود.

ج)

<div dir="ltr">

```python
norms = np.linalg.norm(X, axis=1, keepdims=True)   # shape (4, 1)
X_unit = X / norms
```

</div>

بدون <code dir="ltr">keepdims</code>، <code dir="ltr">norms</code> شکل <code dir="ltr">(4,)</code> دارد و <code dir="ltr">(4, 2) / (4,)</code> طبق قاعده‌ی broadcasting جور نیست (<code dir="ltr">2</code> با <code dir="ltr">4</code>): خطای <code dir="ltr">ValueError</code>.

د) <code dir="ltr">sd</code> آن ستون صفر است و تقسیم بر صفر <code dir="ltr">nan</code> (برای <code dir="ltr">0/0</code>) تولید می‌کند و NumPy فقط هشدار می‌دهد؛ <code dir="ltr">nan</code> بعداً کل محاسبه‌ی مدل را آلوده می‌کند. راه حل رایج: <code dir="ltr">sd = np.where(sd == 0, 1.0, sd)</code> یا حذف ستون ثابت (که هیچ اطلاعاتی ندارد).

</details>

### سؤال ۳ — softmax پایدار روی سطرها

خروجی یک شبکه‌ی عصبی برای <code dir="ltr">n</code> نمونه و <code dir="ltr">k</code> کلاس، ماتریس <code dir="ltr">Z</code> با شکل <code dir="ltr">(n, k)</code> است. softmax هر سطر را به احتمال تبدیل می‌کند. جاهای خالی را کامل کنید:

</div>

```python
def softmax(Z):
    Z = Z - Z.max(axis=______, keepdims=______)
    E = np.exp(Z)
    return E / E.sum(axis=______, keepdims=True)

Z = np.array([[1.0, 2.0, 3.0],
              [1000.0, 1001.0, 1002.0],
              [0.0, 0.0, 0.0]])
P = softmax(Z)
```

<div dir="rtl">

الف) جاهای خالی را کامل کنید و <code dir="ltr">P</code> را تا ۴ رقم اعشار بنویسید. چرا سطر اول و دوم یکسان‌اند؟

ب) اگر خط تفریق بیشینه را حذف کنیم، خروجی سطر دوم چه می‌شود و چرا؟ (در جلسه‌ی ۸ همین مشکل را برای یک لیست دیدید.)

ج) با یک خط، پیش‌بینی هر نمونه و احتمال کلاس پیش‌بینی‌شده را حساب کنید.

د) **cross-entropy** میانگین <code dir="ltr">-log(P[i, y&#95;i])</code> روی نمونه‌هاست. با برچسب‌های <code dir="ltr">y = np.array([2, 0, 1])</code> آن را **بدون حلقه** حساب کنید (راهنمایی: اندیس‌گذاری فانتزی با دو آرایه‌ی اندیس).

<details>
<summary><b>پاسخ</b></summary>

الف) <code dir="ltr">axis=1</code>، <code dir="ltr">keepdims=True</code>، <code dir="ltr">axis=1</code>.

<div dir="ltr">

```text
[[0.09   0.2447 0.6652]
 [0.09   0.2447 0.6652]
 [0.3333 0.3333 0.3333]]
```

</div>

softmax به **اضافه کردن یک عدد ثابت** به همه‌ی ورودی‌های یک سطر حساس نیست (<code dir="ltr">exp(z + c) = exp(z) exp(c)</code> و <code dir="ltr">exp(c)</code> از صورت و مخرج حذف می‌شود)؛ سطر دوم همان سطر اول به‌علاوه‌ی ۹۹۹ است.

ب) <code dir="ltr">np.exp(1000)</code> در <code dir="ltr">float</code> سرریز می‌کند و <code dir="ltr">inf</code> می‌شود؛ <code dir="ltr">inf / inf</code> برابر <code dir="ltr">nan</code> است و سطر دوم <code dir="ltr">[nan nan nan]</code> می‌شود (با هشدار). کم کردن بیشینه، بزرگ‌ترین نما را صفر می‌کند و سرریز رخ نمی‌دهد.

ج) <code dir="ltr">pred = P.argmax(axis=1)</code> و <code dir="ltr">conf = P.max(axis=1)</code>.

د)

<div dir="ltr">

```python
y = np.array([2, 0, 1])
ce = -np.log(P[np.arange(len(y)), y]).mean()
```

</div>

<code dir="ltr">P[np.arange(3), y]</code> عضوهای <code dir="ltr">P[0, 2]</code>، <code dir="ltr">P[1, 0]</code> و <code dir="ltr">P[2, 1]</code> را برمی‌دارد: <code dir="ltr">[0.6652, 0.09, 0.3333]</code>؛ حاصل <code dir="ltr">ce ≈ 1.3046</code>. (در عمل <code dir="ltr">np.clip(P, 1e-12, 1)</code> قبل از <code dir="ltr">log</code> از <code dir="ltr">log(0)</code> جلوگیری می‌کند.)

</details>

### سؤال ۴ — ماتریس فاصله و kNN برداری

</div>

```python
Xtr = np.array([[1.0, 1.0], [1.0, 2.0], [5.0, 5.0], [6.0, 5.0], [5.0, 6.0]])
ytr = np.array([0, 0, 1, 1, 1])
Xte = np.array([[2.0, 2.0], [4.0, 4.0]])

diff = Xte[:, np.newaxis, :] - Xtr[np.newaxis, :, :]
D = np.sqrt((diff ** 2).sum(axis=2))
nearest = np.argsort(D, axis=1)[:, :3]
pred = (ytr[nearest].mean(axis=1) > 0.5).astype(int)
```

<div dir="rtl">

الف) شکل <code dir="ltr">diff</code>، <code dir="ltr">D</code> و <code dir="ltr">nearest</code> چیست؟ <code dir="ltr">D[i, j]</code> چه معنایی دارد؟

ب) <code dir="ltr">D</code> (تا ۳ رقم اعشار)، <code dir="ltr">nearest</code>، <code dir="ltr">ytr[nearest]</code> و <code dir="ltr">pred</code> را بنویسید.

ج) این کد همان kNN جلسه‌ی ۸ است که آنجا با دو حلقه‌ی تودرتو نوشتیم. اگر <code dir="ltr">n&#95;test = 10&#95;000</code>، <code dir="ltr">n&#95;train = 50&#95;000</code> و <code dir="ltr">d = 100</code> باشد، آرایه‌ی <code dir="ltr">diff</code> چند عدد اعشاری دارد؟ با ۸ بایت برای هر عدد، چقدر حافظه لازم است؟ چه راه حلی پیشنهاد می‌کنید؟

د) با استفاده از اتحاد <code dir="ltr">‖a − b‖² = ‖a‖² + ‖b‖² − 2 a·b</code>، مربع فاصله‌ها را با یک ضرب ماتریسی بنویسید.

<details>
<summary><b>پاسخ</b></summary>

الف) <code dir="ltr">diff</code>: <code dir="ltr">(2, 5, 2)</code>؛ <code dir="ltr">D</code>: <code dir="ltr">(2, 5)</code>؛ <code dir="ltr">nearest</code>: <code dir="ltr">(2, 3)</code>. <code dir="ltr">D[i, j]</code> فاصله‌ی نمونه‌ی آزمون <code dir="ltr">i</code> از نمونه‌ی آموزش <code dir="ltr">j</code> است.

ب)

<div dir="ltr">

```text
D = [[1.414 1.    4.243 5.    5.   ]
     [4.243 3.606 1.414 2.236 2.236]]
nearest = [[1 0 2]
           [2 3 4]]
ytr[nearest] = [[0 0 1]
                [1 1 1]]
pred = [0 1]
```

</div>

(در سطر دوم دو فاصله‌ی برابر <code dir="ltr">2.236</code> وجود دارد؛ <code dir="ltr">argsort</code> در حالت تساوی ترتیب اندیس‌ها را حفظ می‌کند.)

ج) <code dir="ltr">10&#95;000 × 50&#95;000 × 100 = 5 × 10¹⁰</code> عدد، یعنی حدود ۴۰۰ گیگابایت؛ بسیار بیشتر از حافظه‌ی هر کامپیوتر معمولی. راه حل: داده‌ی آزمون را در **دسته‌های** کوچک (مثلاً ۱۰۰ تایی، با یک generator مثل جلسه‌ی ۱۴) پردازش کنید یا از اتحاد قسمت (د) استفاده کنید که آرایه‌ی سه‌بعدی نمی‌سازد.

د)

<div dir="ltr">

```python
sq = (Xte ** 2).sum(axis=1)[:, np.newaxis] + (Xtr ** 2).sum(axis=1)[np.newaxis, :] - 2 * Xte @ Xtr.T
D = np.sqrt(np.maximum(sq, 0))
```

</div>

بزرگ‌ترین آرایه‌ی این روش <code dir="ltr">(n&#95;test, n&#95;train)</code> است. <code dir="ltr">np.maximum(sq, 0)</code> جلوی عددهای منفی بسیار کوچک ناشی از خطای گرد کردن را می‌گیرد.

</details>

### سؤال ۵ — رگرسیون خطی: معادلات نرمال و گرادیان کاهشی

</div>

```python
x = np.array([0.0, 1.0, 2.0, 3.0, 4.0])
y = np.array([1.0, 3.0, 5.5, 7.0, 9.5])
X = np.column_stack([np.ones_like(x), x])     # shape (5, 2)

w_ne = np.linalg.solve(X.T @ X, X.T @ y)
w_ls = np.linalg.lstsq(X, y, rcond=None)[0]

w = np.zeros(2)
for epoch in range(2000):
    grad = 2 / len(y) * X.T @ (X @ w - y)
    w = w - 0.05 * grad
```

<div dir="rtl">

الف) <code dir="ltr">X</code> چه شکلی دارد و ستون اول آن (همه یک) چه نقشی دارد؟ <code dir="ltr">w&#95;ne</code> چیست؟ (عرض از مبدأ و شیب.)

ب) گرادیان در **اولین** تکرار (با <code dir="ltr">w = [0, 0]</code>) چیست؟ بعد از ۲۰۰۰ تکرار، <code dir="ltr">w</code> و <code dir="ltr">grad</code> تقریباً چه می‌شوند؟

ج) فرمول <code dir="ltr">grad = 2/n · Xᵀ (X w − y)</code> را با حلقه‌ی گرادیان کاهشی کلاس <code dir="ltr">LinearRegression</code> جلسه‌ی ۱۸ مقایسه کنید. چرا نسخه‌ی برداری برای هر تعداد ویژگی بدون تغییر کار می‌کند؟

د) دانشجویی <code dir="ltr">y</code> را به شکل <code dir="ltr">y.reshape(-1, 1)</code> نوشته اما <code dir="ltr">w</code> را یک‌بعدی گذاشته است. <code dir="ltr">X @ w - y</code> چه شکلی دارد و MSE چه عددی می‌شود؟ (برای <code dir="ltr">w = w&#95;ne</code>.)

ه) معادلات نرمال جواب را در یک قدم می‌دهند. چرا در شبکه‌های عصبی و داده‌های بسیار بزرگ باز هم از گرادیان کاهشی استفاده می‌شود؟

<details>
<summary><b>پاسخ</b></summary>

الف) <code dir="ltr">(5, 2)</code>. ستون یک‌ها باعث می‌شود اولین وزن نقش **عرض از مبدأ (bias)** را داشته باشد: <code dir="ltr">ŷ = w₀ · 1 + w₁ · x</code>. <code dir="ltr">w&#95;ne = w&#95;ls = [1. , 2.1]</code>؛ یعنی <code dir="ltr">ŷ = 1 + 2.1 x</code> و MSE آن <code dir="ltr">0.04</code> است.

ب) در شروع <code dir="ltr">X w = 0</code>؛ پس <code dir="ltr">grad = -2/5 · Xᵀ y = [-10.4, -29.2]</code> (منفی یعنی وزن‌ها باید زیاد شوند). بعد از ۲۰۰۰ تکرار <code dir="ltr">w ≈ [1. , 2.1]</code> (همان جواب معادلات نرمال) و <code dir="ltr">grad ≈ [0, 0]</code>؛ در کمینه گرادیان صفر است.

ج) در جلسه‌ی ۱۸ برای هر نمونه خطا را حساب و در حلقه جمع می‌کردیم؛ <code dir="ltr">Xᵀ (X w − y)</code> همان جمع را برای **همه‌ی وزن‌ها هم‌زمان** حساب می‌کند: عضو <code dir="ltr">j</code> آن <code dir="ltr">Σᵢ xᵢⱼ (ŷᵢ − yᵢ)</code> است. چون همه‌چیز با شکل‌ها بیان شده، با <code dir="ltr">d = 2</code> یا <code dir="ltr">d = 1000</code> همین یک خط کار می‌کند.

د) <code dir="ltr">X @ w</code> شکل <code dir="ltr">(5,)</code> دارد و <code dir="ltr">y</code> شکل <code dir="ltr">(5, 1)</code>؛ حاصل <code dir="ltr">(5, 5)</code> است و MSE عدد بی‌معنی <code dir="ltr">17.68</code> می‌شود (به جای <code dir="ltr">0.04</code>). هیچ خطایی رخ نمی‌دهد؛ گرادیان هم با این شکل‌ها غلط حساب می‌شود و مدل درست یاد نمی‌گیرد.

ه) معادلات نرمال به ساختن و حل <code dir="ltr">Xᵀ X</code> با شکل <code dir="ltr">(d, d)</code> نیاز دارند (هزینه‌ی تقریباً <code dir="ltr">d³</code>) و کل داده باید یک‌جا در حافظه باشد؛ برای میلیون‌ها ویژگی یا نمونه عملی نیست. مهم‌تر اینکه برای مدل‌های **غیرخطی** (شبکه‌های عصبی) اصلاً فرمول بسته‌ای وجود ندارد؛ گرادیان کاهشی (معمولاً روی دسته‌های کوچک داده) تنها راه عمومی است.

</details>

### سؤال ۶ — لایه‌ی Dense برداری

یک لایه‌ی کاملاً متصل (Dense) با ۳ ورودی و ۲ خروجی، برای یک دسته (batch) از ۴ نمونه:

</div>

```python
X = np.array([[1.0, 0.0, 2.0],
              [0.0, 1.0, 1.0],
              [2.0, 2.0, 0.0],
              [1.0, 1.0, 1.0]])
W = np.array([[1.0, -1.0],
              [0.0, 2.0],
              [1.0, 0.0]])
b = np.array([0.0, -1.0])

H = X @ W + b
A = np.maximum(H, 0)
```

<div dir="rtl">

الف) شکل <code dir="ltr">X</code>، <code dir="ltr">W</code>، <code dir="ltr">b</code>، <code dir="ltr">H</code> را بنویسید. در <code dir="ltr">X @ W + b</code> کجا broadcasting رخ می‌دهد؟

ب) <code dir="ltr">H</code> و <code dir="ltr">A</code> را حساب کنید.

ج) این لایه چند پارامتر آموختنی دارد؟ اگر لایه‌ی بعدی ۲ ورودی و ۱ خروجی داشته باشد، کل شبکه چند پارامتر دارد؟

د) در جلسه‌ی ۱۹ کلاس <code dir="ltr">Dense</code> را با حلقه روی نورون‌ها نوشتیم. متد <code dir="ltr">forward</code> را با یک خط NumPy بازنویسی کنید. اگر کسی <code dir="ltr">W</code> را با شکل <code dir="ltr">(2, 3)</code> بسازد، کد شما کجا و با چه پیامی خطا می‌دهد؟ با یک <code dir="ltr">raise</code> (جلسه‌ی ۱۵) پیام واضح‌تری بدهید.

<details>
<summary><b>پاسخ</b></summary>

الف) <code dir="ltr">X</code>: <code dir="ltr">(4, 3)</code>، <code dir="ltr">W</code>: <code dir="ltr">(3, 2)</code>، <code dir="ltr">b</code>: <code dir="ltr">(2,)</code>، <code dir="ltr">H</code>: <code dir="ltr">(4, 2)</code>. حاصل <code dir="ltr">X @ W</code> شکل <code dir="ltr">(4, 2)</code> دارد و <code dir="ltr">b</code> با شکل <code dir="ltr">(2,)</code> به **هر سطر** (هر نمونه) اضافه می‌شود.

ب)

<div dir="ltr">

```text
H = [[ 3. -2.]
     [ 1.  1.]
     [ 2.  1.]
     [ 2.  0.]]
A = [[3. 0.]
     [1. 1.]
     [2. 1.]
     [2. 0.]]
```

</div>

ج) <code dir="ltr">3 × 2 + 2 = 8</code> پارامتر. لایه‌ی دوم <code dir="ltr">2 × 1 + 1 = 3</code> پارامتر دارد؛ کل شبکه <code dir="ltr">11</code>. (تعداد پارامترها به اندازه‌ی دسته بستگی ندارد.)

د)

<div dir="ltr">

```python
def forward(self, X):
    if X.shape[1] != self.W.shape[0]:
        raise ValueError(f"expected {self.W.shape[0]} features, got {X.shape[1]}")
    return X @ self.W + self.b
```

</div>

بدون بررسی، <code dir="ltr">X @ W</code> با <code dir="ltr">(4, 3) @ (2, 3)</code> خطای <code dir="ltr">ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0 ...</code> می‌دهد که مفید است اما نمی‌گوید کدام لایه مشکل دارد؛ پیام اختصاصی، اشکال‌زدایی یک شبکه‌ی چندلایه را بسیار ساده‌تر می‌کند.

</details>

### سؤال ۷ — دسته‌های کوچک با مولد تصادفی

تابع زیر داده را در هر epoch به هم می‌ریزد و دسته‌های کوچک (mini-batch) تولید می‌کند:

</div>

```python
def batches(X, y, batch_size, rng):
    idx = rng.permutation(len(y))
    for start in range(0, len(y), batch_size):
        b = idx[start:start + batch_size]
        yield X[b], y[b]

X = np.arange(20).reshape(10, 2)
y = np.arange(10)
rng = np.random.default_rng(0)
for epoch in range(2):
    sizes = [len(yb) for _, yb in batches(X, y, 4, rng)]
    print(epoch, sizes)
```

<div dir="rtl">

الف) خروجی چیست؟ برای <code dir="ltr">rng = np.random.default&#95;rng(0)</code>، اولین جای‌گشت <code dir="ltr">[4 6 2 7 3 5 9 0 8 1]</code> است. سه دسته‌ی <code dir="ltr">y</code> در epoch اول چیست؟

ب) آیا ترتیب داده در epoch دوم با epoch اول یکسان است؟ چرا این خوب است؟ اگر بخواهیم کل آزمایش (هر دو epoch) با اجرای دوباره‌ی برنامه دقیقاً تکرار شود، چه باید کرد؟

ج) دانشجویی به جای <code dir="ltr">permutation</code> نوشته است <code dir="ltr">rng.shuffle(X); rng.shuffle(y)</code>. دو اشکال این کار را بگویید.

د) این تابع را با تابع <code dir="ltr">batches</code> جلسه‌ی ۱۴ (که روی لیست کار می‌کرد) مقایسه کنید. چه چیزی NumPy را اینجا مناسب‌تر می‌کند؟

<details>
<summary><b>پاسخ</b></summary>

الف)

<div dir="ltr">

```text
0 [4, 4, 2]
1 [4, 4, 2]
```

</div>

دسته‌های <code dir="ltr">y</code> در epoch اول: <code dir="ltr">[4 6 2 7]</code>، <code dir="ltr">[3 5 9 0]</code>، <code dir="ltr">[8 1]</code> (دسته‌ی آخر ناقص است، مثل <code dir="ltr">drop&#95;last=False</code>).

ب) نه؛ چون <code dir="ltr">rng</code> بین دو epoch جلو رفته، جای‌گشت دوم متفاوت است (<code dir="ltr">[2 9 3 6 0 4 8 7 5 1]</code>). این مطلوب است: اگر ترتیب ثابت باشد، مدل هر بار همان دسته‌ها را با همان ترتیب می‌بیند و گرادیان‌ها همبستگی پیدا می‌کنند. برای تکرارپذیری کل آزمایش کافی است مولد **یک بار** با <code dir="ltr">seed</code> ثابت در ابتدای برنامه ساخته شود (همان کاری که این کد کرده است)؛ نباید در هر epoch دوباره با همان <code dir="ltr">seed</code> ساخته شود، وگرنه همه‌ی epochها ترتیب یکسان خواهند داشت.

ج) (۱) <code dir="ltr">X</code> و <code dir="ltr">y</code> جداگانه به هم ریخته می‌شوند و جفت نمونه و برچسب از بین می‌رود (هر کدام جای‌گشت متفاوتی می‌گیرند). (۲) <code dir="ltr">shuffle</code> درجاست و آرایه‌های **اصلی** را تغییر می‌دهد؛ اگر جای دیگری از برنامه (مثلاً ارزیابی) به ترتیب اصلی وابسته باشد، خراب می‌شود.

د) با NumPy یک دسته با **اندیس‌گذاری فانتزی** (<code dir="ltr">X[b]</code>) در یک قدم ساخته می‌شود، به جای ساختن لیست با حلقه، و خروجی مستقیماً یک ماتریس <code dir="ltr">(batch&#95;size, n&#95;features)</code> است که می‌تواند وارد <code dir="ltr">X @ W + b</code> (سؤال ۶) شود. ساختار generator (یک دسته در هر لحظه) همان است.

</details>

</div>

<div dir="rtl">

## خلاصه‌ی جلسه

- برداری‌سازی یعنی محاسبه روی کل آرایه به جای حلقه؛ ufuncها، مقایسه‌ها، <code dir="ltr">np.where</code> و <code dir="ltr">np.clip</code> همه عضو به عضو کار می‌کنند.
- broadcasting شکل‌ها را از راست مقایسه می‌کند (برابر یا ۱)؛ همیشه شکل حاصل را پیش‌بینی کنید و مراقب <code dir="ltr">(n, 1)</code> در برابر <code dir="ltr">(n,)</code> باشید.
- <code dir="ltr">axis</code> مشخص می‌کند کدام بُعد جمع شود و <code dir="ltr">keepdims</code> نتیجه را برای broadcasting آماده نگه می‌دارد.
- <code dir="ltr">np.random.default&#95;rng(seed)</code> اعداد تصادفی تکرارپذیر می‌دهد؛ <code dir="ltr">X</code> و <code dir="ltr">y</code> را با یک جای‌گشت مشترک به هم بریزید.
- <code dir="ltr">@</code> ضرب ماتریسی است؛ دستگاه را با <code dir="ltr">solve</code> حل کنید نه <code dir="ltr">inv</code>؛ <code dir="ltr">lstsq</code>، <code dir="ltr">norm</code> و <code dir="ltr">eigh</code> ابزارهای اصلی مدل‌های خطی‌اند.

**جلسه‌ی بعد:** Pandas ۱؛ کار با داده‌ی جدولی با <code dir="ltr">Series</code> و <code dir="ltr">DataFrame</code>، خواندن فایل CSV و Excel، و انتخاب و فیلتر کردن داده.

</div>